# Differentiable Compound Optics — a toy end-to-end camera-design demo



## 0. Setup



In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

def show(imgs, titles, cmap=None, figsize=None):
    n = len(imgs)
    plt.figure(figsize=figsize or (3*n, 3))
    for i,(im,t) in enumerate(zip(imgs, titles)):
        plt.subplot(1, n, i+1)
        a = im.detach().cpu().numpy() if torch.is_tensor(im) else im
        a = np.clip(a, 0, 1)
        plt.imshow(a, cmap=cmap); plt.title(t, fontsize=10); plt.axis("off")
    plt.tight_layout(); plt.show()


## 1. Creating Synthetic Scene



In [ ]:
IMG = 60  # image size (small on purpose)

def make_scene(batch=1):
    """Synthetic 'scenes': edges, dots and bars — content that reveals blur."""
    scenes = []
    for _ in range(batch):
        img = torch.zeros(3, IMG, IMG)
        # colored background gradient
        yy, xx = torch.meshgrid(torch.linspace(0,1,IMG), torch.linspace(0,1,IMG), indexing="ij")
        img[0] = 0.15 + 0.10*xx; img[1] = 0.15 + 0.10*yy; img[2] = 0.20
        # a few bright bars / dots of random color & position (high-frequency detail)
        for _ in range(6):
            c = torch.rand(3)
            if torch.rand(1) < 0.5:  # bar
                x0 = np.random.randint(4, IMG-12); w = np.random.randint(2,4)
                img[:, :, x0:x0+w] = c[:,None,None]
            else:  # dot
                y0 = np.random.randint(6, IMG-6); x0 = np.random.randint(6, IMG-6)
                img[:, y0-2:y0+2, x0-2:x0+2] = c[:,None,None]
        scenes.append(img.clamp(0,1))
    return torch.stack(scenes).to(device)

scene = make_scene(1)[0]
show([scene.permute(1,2,0)], ["A synthetic scene"])


## 2. Compound optics & the PSF - Toy Physics Simulation



In [ ]:
KS = 11                       # PSF kernel size
_coords = torch.arange(KS) - KS//2
_gy, _gx = torch.meshgrid(_coords.float(), _coords.float(), indexing="ij")
_gy, _gx = _gy.to(device), _gx.to(device)

def make_psf(widths, shifts):
    """ RGB PSF from per-channel (width, shift) optics params.
       widths:  (3,) gaussian sigma per channel  -> blur amount (aberration)
       shifts:  (3,) lateral shift per channel    -> chromatic aberration
       Returns a (3, KS, KS) PSF, each channel normalized to sum 1 (energy preserving)."""
    psfs = []
    for c in range(3):
        sig = torch.clamp(widths[c], 0.35, 4.0)
        cx  = shifts[c]                       # shift only the x-axis for simplicity
        g = torch.exp(-(((_gx-cx)**2 + _gy**2) / (2*sig**2)))
        g = g / (g.sum() + 1e-8)              # energy preserving (paper: unit-sum PSF)
        psfs.append(g)
    return torch.stack(psfs)                  # (3, KS, KS)

def apply_optics(img, psf):
    """Convolve each RGB channel of img (3,H,W) with its PSF channel."""
    x = img.unsqueeze(0)                       # (1,3,H,W)
    k = psf.unsqueeze(1)                       # (3,1,KS,KS)  depthwise
    out = F.conv2d(x, k, padding=KS//2, groups=3)
    return out.squeeze(0).clamp(0,1)

# Visualize a "bad" aberrated lens vs a "good" one
bad  = (torch.tensor([3.2,2.6,3.0], device=device), torch.tensor([ 2.5,0.0,-2.5], device=device))
good = (torch.tensor([0.5,0.5,0.5], device=device), torch.tensor([ 0.0,0.0, 0.0], device=device))

psf_bad, psf_good = make_psf(*bad), make_psf(*good)
show([psf_bad.permute(1,2,0)/psf_bad.max(), psf_good.permute(1,2,0)/psf_good.max()],
     ["PSF: aberrated lens (wide + chromatic)", "PSF: well-corrected lens"])
show([apply_optics(scene, psf_bad).permute(1,2,0), apply_optics(scene, psf_good).permute(1,2,0)],
     ["Scene through aberrated lens", "Scene through good lens"])


## 3. The optics meta-network



In [ ]:
class OpticsMetaNet(nn.Module):
    """MLP surrogate: optics params (6 numbers) -> RGB PSF (3*KS*KS).
       Stand-in for the paper's MLP-encoder + conv-decoder that replaces Zemax."""
    def __init__(self, n_params=6, ks=KS):
        super().__init__()
        self.ks = ks
        self.net = nn.Sequential(
            nn.Linear(n_params, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 3*ks*ks),
        )
    def forward(self, params):
        raw = self.net(params).view(-1, 3, self.ks, self.ks)
        # softmax over each PSF
        flat = raw.view(-1, 3, self.ks*self.ks).softmax(dim=-1)
        return flat.view(-1, 3, self.ks, self.ks).squeeze(0)

# --- Build a training set by querying the "simulator" make_psf on random designs ---
def sample_params(n):
    w = torch.rand(n,3)*3.4 + 0.4
    s = (torch.rand(n,3)-0.5)*6.0
    return torch.cat([w, s], dim=1).to(device)

def gt_psf_batch(params):
    out = [make_psf(p[:3], p[3:]) for p in params]
    return torch.stack(out)

print("Training the optics meta-network to imitate the lens simulator...")
meta = OpticsMetaNet().to(device)
opt  = torch.optim.Adam(meta.parameters(), lr=2e-3)
train_p = sample_params(4000)
train_y = gt_psf_batch(train_p)
for step in range(1500):
    idx = torch.randint(0, train_p.shape[0], (256,), device=device)
    p, y = train_p[idx], train_y[idx]
    pred = torch.stack([meta(pi) for pi in p])
    loss = F.mse_loss(pred, y)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 300 == 0: print(f"  step {step:4d}  MSE = {loss.item():.2e}")

# Check: meta-network PSF vs ground-truth simulator PSF on a held-out design
test_p = sample_params(1)[0]
mp, gp = meta(test_p), make_psf(test_p[:3], test_p[3:])
show([gp.permute(1,2,0)/gp.max(), mp.permute(1,2,0)/mp.max()],
     ["Ground-truth (simulator) PSF", "Meta-network PSF (learned)"])
print("The network now reproduces the simulator and is fully differentiable.")


## 3.1 Module Playground: Team A
Design your optics parameters and test the ground truth PSF against the meta-network PSF!

In [ ]:
# Team A: Enter your custom optics parameters here!
# Format: [width_R, width_G, width_B, shift_R, shift_G, shift_B]
# Widths should ideally be between 0.35 and 4.0. Shifts can be around -3.0 to 3.0.
custom_params_A = torch.tensor([1, 1, 1, 2.0, 2.0, -2.0], device=device)

# Generate PSFs
meta_psf_A = meta(custom_params_A)
gt_psf_A = make_psf(custom_params_A[:3], custom_params_A[3:])

# Visualize
show([gt_psf_A.permute(1,2,0)/gt_psf_A.max(), meta_psf_A.permute(1,2,0)/meta_psf_A.max()],
     ["Team A: Ground-truth PSF", "Team A: Meta-network PSF"])

## 3.2 Module Playground: Team B
Design your optics parameters and test the ground truth PSF against the meta-network PSF!

In [ ]:
# Team B: Enter your custom optics parameters here!
# Format: [width_R, width_G, width_B, shift_R, shift_G, shift_B]
# Widths should ideally be between 0.35 and 4.0. Shifts can be around -3.0 to 3.0.
custom_params_B = torch.tensor([2.5, 2, 3.0, 2.0, 1.0, -1.0], device=device)

# Generate PSFs
meta_psf_B = meta(custom_params_B)
gt_psf_B = make_psf(custom_params_B[:3], custom_params_B[3:])

# Visualize
show([gt_psf_B.permute(1,2,0)/gt_psf_B.max(), meta_psf_B.permute(1,2,0)/meta_psf_B.max()],
     ["Team B: Ground-truth PSF", "Team B: Meta-network PSF"])

## 4. The differentiable sensor



In [ ]:
def sensor(img, exposure=400.0, gain=1.0/400.0, read_noise=0.5, train=True):
    """Differentiable sensor: img in [0,1] radiance -> noisy readout in ~[0,1]."""
    photons = img * exposure                                  # mean photon count
    if train:
        # differentiable Poisson approx: N(mu, mu)  (shot noise)
        electrons = photons + torch.sqrt(photons + 1e-6) * torch.randn_like(photons)
    else:
        electrons = torch.poisson(photons.clamp(min=0))
    electrons = electrons + read_noise * torch.randn_like(electrons)   # read/dark noise
    electrons = torch.clamp(electrons, 0, exposure*1.2)               # well capacity
    digital   = electrons * gain                                      # gain K
    digital   = digital + (torch.rand_like(digital) - 0.5) * gain     # quantization noise
    return digital.clamp(0, 1)

clean = apply_optics(scene, psf_good)
show([sensor(clean, exposure=2000., gain=1/2000., train=False).permute(1,2,0),
      sensor(clean, exposure=60.,   gain=1/60.,   train=False).permute(1,2,0)],
     ["Sensor readout — bright (lots of photons)", "Sensor readout — low light (noisy!)"])


## 5. The ISP + NN



### Team A: Mini U-Net ISP Module

In [ ]:
class TinyISP_TeamA(nn.Module):
    """Mini U-Net software ISP: learns to denoise/deblur the sensor RAW into a clean image."""
    def __init__(self):
        super().__init__()
        # Encoder
        self.enc1 = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(inplace=True))
        self.pool = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(inplace=True))

        # Decoder
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec1 = nn.Sequential(nn.Conv2d(32 + 16, 16, 3, padding=1), nn.ReLU(inplace=True))
        self.out = nn.Conv2d(16, 3, 3, padding=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))

        d1 = self.up(e2)
        d1 = torch.cat([d1, e1], dim=1) # Skip connection

        out = self.dec1(d1)
        out = self.out(out)

        # Residual learning
        return (x + out).clamp(0,1)

### Team B: Regular CNN ISP Module

In [ ]:
class TinyISP_TeamB(nn.Module):
    """Regular CNN software ISP."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 3, 3, padding=1),
        )

    def forward(self, x):
        # Residual learning
        return (x + self.net(x)).clamp(0,1)

### ISP Selection & Pipeline Configuration

In [ ]:
# ---> SET WHICH TEAM'S ISP TO USE HERE <---
TinyISP = TinyISP_TeamA  # Change to TinyISP_TeamB to test the regular CNN!

def pipeline(scene, optics_params, isp, exposure, train=True):
    """Stage 1..4: scene -> optics(meta-net PSF) -> sensor -> ISP -> output."""
    psf   = meta(optics_params)                     # differentiable optics (stage 2)
    optic = apply_optics(scene, psf)                # convolve with PSF
    raw   = sensor(optic, exposure, 1.0/exposure, train=train)   # sensor (stage 3)
    out   = isp(raw.unsqueeze(0)).squeeze(0)        # ISP (stage 4)
    return out, psf

print(f"Pipeline ready. Currently using: {TinyISP.__name__}")

## 6. Proximal Compositional Optimization



In [ ]:


def proximal_compositional_optimize(
        init_optics, exposure, n_cycles=12, steps_per_block=8, n_finetune=40,
        beta=2.0, lr_optics=0.08, lr_isp=3e-3, optimize_optics=True, verbose=True):
    """Algorithm: round-robin block updates + proximal reg, then joint fine-tune."""

    optics = init_optics.clone().detach().to(device).requires_grad_(optimize_optics)
    isp = TinyISP().to(device)

    opt_optics = torch.optim.Adam([optics], lr=lr_optics) if optimize_optics else None
    opt_isp    = torch.optim.Adam(isp.parameters(), lr=lr_isp)

    def task_loss(out, target):
        l1 = F.l1_loss(out, target)
        gx = lambda t: t[:,:,1:]-t[:,:,:-1]; gy = lambda t: t[:,1:,:]-t[:,:-1,:]
        edge = F.l1_loss(gx(out), gx(target)) + F.l1_loss(gy(out), gy(target))
        return l1 + 0.5*edge

    blocks = [("isp", opt_isp)]
    if optimize_optics: blocks = [("optics", opt_optics)] + blocks

    loss_history = []

    for cyc in range(n_cycles):
        for name, optimizer in blocks:
            for _ in range(steps_per_block):
                s = make_scene(4)
                prev = optics.detach().clone()
                loss = 0.

                for b in range(s.shape[0]):
                    out,_ = pipeline(s[b], optics, isp, exposure, train=True)
                    loss = loss + task_loss(out, s[b])

                loss = loss / s.shape[0]

                if name == "optics" and optimize_optics:
                    loss = loss + beta * ((optics - prev)**2).sum()

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        loss_history.append(loss.item())

        if verbose and cyc % 3 == 0:
            print(f"  cycle {cyc:2d}  task loss = {loss.item():.4f}")

    params = (list(isp.parameters()) + ([optics] if optimize_optics else []))
    ft = torch.optim.Adam(params, lr=lr_isp)

    for _ in range(n_finetune):
        s = make_scene(4); loss = 0.
        for b in range(s.shape[0]):
            out,_ = pipeline(s[b], optics, isp, exposure, train=True)
            loss = loss + task_loss(out, s[b])

        (loss/s.shape[0]).backward()
        ft.step()
        ft.zero_grad()

    return optics.detach(), isp, loss_history

print("Proximal Optimizer defined.")

## 7. Metrics & Real Data Setup

In [ ]:
!pip install -q lpips torchmetrics
import torchvision
import torchvision.transforms as T
import lpips
from torchmetrics.image import StructuralSimilarityIndexMeasure
import math

print("Setting up Metrics and CIFAR-10...")
loss_fn_vgg = lpips.LPIPS(net='vgg').to(device)
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

def compute_metrics(out, target):
    mse = F.mse_loss(out, target)
    psnr_val = 10 * math.log10(1.0 / (mse.item() + 1e-10))
    ssim_val = ssim_metric(out.unsqueeze(0), target.unsqueeze(0)).item()
    lpips_val = loss_fn_vgg((out.unsqueeze(0)*2)-1, (target.unsqueeze(0)*2)-1).item()
    return psnr_val, ssim_val, lpips_val

transform = T.Compose([T.Resize((IMG, IMG)), T.ToTensor()])
cifar = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
cifar_loader = torch.utils.data.DataLoader(cifar, batch_size=64, shuffle=True)
cifar_iter = iter(cifar_loader)

def make_scene_real(batch=1):
    global cifar_iter
    try:
        images, _ = next(cifar_iter)
    except StopIteration:
        cifar_iter = iter(cifar_loader)
        images, _ = next(cifar_iter)
    return images[:batch].to(device)

## 8. Training Models

In [ ]:
# ==========================================
# HYPERPARAMETERS & CONFIGURATION
# ==========================================
EXPOSURE = 45.0  # Simulated sensor exposure/gain

# Number of optimization cycles for each stage
CYCLES_NOMINAL = 50       # Baseline training on synthetic data
CYCLES_NOMINAL_REAL = 50  # Baseline training on CIFAR-10 real data
CYCLES_E2E_SYNTH = 45    # End-to-end training on synthetic data
CYCLES_E2E_REAL = 45     # End-to-end training on CIFAR-10 real data

# Learning Rates
LR_OPTICS = 0.08
LR_ISP = 3e-3

# Initial Optics Parameters [width_R, width_G, width_B, shift_R, shift_G, shift_B]
init_nominal_optics = torch.tensor([0.2, 0.3, 0.5,  0.1, 0.0, -0.0], device=device)
init_e2e_optics = torch.tensor([0.4, 0.3, 0.8,  0.1, 0.0, -0.1], device=device)
# ==========================================

if not callable(make_scene):
    print("⚠️ `make_scene` is currently a tensor. Please re-run Cell 1 (Creating Synthetic Scene) to restore the function!")
else:
    print("=== Training NOMINAL (Synthetic) ===")
    nom_optics, nom_isp, nom_losses = proximal_compositional_optimize(
        init_nominal_optics, EXPOSURE, optimize_optics=False,
        n_cycles=CYCLES_NOMINAL, lr_optics=LR_OPTICS, lr_isp=LR_ISP
    )

    print("\n=== Training END-TO-END (Synthetic) ===")
    e2e_optics, e2e_isp, e2e_losses = proximal_compositional_optimize(
        init_e2e_optics, EXPOSURE, optimize_optics=True,
        n_cycles=CYCLES_E2E_SYNTH, lr_optics=LR_OPTICS, lr_isp=LR_ISP
    )

    print("\n=== Training NOMINAL (CIFAR-10 Real Images) ===")
    original_make_scene = make_scene
    make_scene = make_scene_real
    nom_optics_real, nom_isp_real, nom_real_losses = proximal_compositional_optimize(
        init_nominal_optics, EXPOSURE, optimize_optics=False,
        n_cycles=CYCLES_NOMINAL_REAL, lr_optics=LR_OPTICS, lr_isp=LR_ISP
    )
    make_scene = original_make_scene

    print("\n=== Training END-TO-END (CIFAR-10 Real Images) ===")
    original_make_scene = make_scene
    make_scene = make_scene_real
    e2e_optics_real, e2e_isp_real, e2e_real_losses = proximal_compositional_optimize(
        init_e2e_optics, EXPOSURE, optimize_optics=True,
        n_cycles=CYCLES_E2E_REAL, lr_optics=LR_OPTICS, lr_isp=LR_ISP
    )
    make_scene = original_make_scene

    # Plot Training Losses
    plt.figure(figsize=(10, 5))
    plt.plot(nom_losses, label="Nominal (Synthetic)", marker='o')
    plt.plot(e2e_losses, label="End-to-End (Synthetic)", marker='s')
    plt.plot(nom_real_losses, label="Nominal (Real CIFAR-10)", marker='v')
    plt.plot(e2e_real_losses, label="End-to-End (Real CIFAR-10)", marker='^')
    plt.title("Training Loss Over Cycles")
    plt.xlabel("Cycle")
    plt.ylabel("Task Loss")
    plt.legend()
    plt.grid(True)
    plt.show()



## 9. Evaluating Models

In [ ]:
if 'nom_losses' not in locals():
    print("Please run the training cell first!")
else:
    print("\n=== Synthetic Evaluation ===")
    torch.manual_seed(123)
    test_scenes = make_scene(40)
    def evaluate(optics, isp):
        ps = [compute_metrics(pipeline(s, optics, isp, EXPOSURE, train=False)[0], s)[0] for s in test_scenes]
        return np.mean(ps)

    nom_psnr, e2e_psnr = evaluate(nom_optics, nom_isp), evaluate(e2e_optics, e2e_isp)
    print(f"Nominal Baseline: {nom_psnr:.2f} dB | End-to-end: {e2e_psnr:.2f} dB")

    demo = make_scene(1)[0]
    with torch.no_grad():
        nom_out, nom_psf = pipeline(demo, nom_optics, nom_isp, EXPOSURE, train=False)
        e2e_out, e2e_psf = pipeline(demo, e2e_optics, e2e_isp, EXPOSURE, train=False)

    show([demo.permute(1,2,0), nom_out.permute(1,2,0), e2e_out.permute(1,2,0)],
         ["Ground-truth", f"Nominal\n{nom_psnr:.2f} dB", f"End-to-end\n{e2e_psnr:.2f} dB"], figsize=(10,3))
    show([nom_psf.permute(1,2,0)/nom_psf.max(), e2e_psf.permute(1,2,0)/e2e_psf.max()],
         ["Nominal PSF", "End-to-end PSF"])

    print("\n=== Evaluation on a random CIFAR-10 Image ===")
    cifar_img = make_scene_real(1)[0]

    with torch.no_grad():
        # Run through CIFAR-trained Nominal Pipeline
        nom_out_img, nom_out_psf = pipeline(cifar_img, nom_optics_real, nom_isp_real, EXPOSURE, train=False)
        nom_psnr_cifar, nom_ssim, nom_lpips = compute_metrics(nom_out_img, cifar_img)

        # Run through End-to-End Real Pipeline
        e2e_out_img, e2e_out_psf = pipeline(cifar_img, e2e_optics_real, e2e_isp_real, EXPOSURE, train=False)
        e2e_psnr_cifar, e2e_ssim, e2e_lpips = compute_metrics(e2e_out_img, cifar_img)

    # Visualize results
    show([cifar_img.permute(1,2,0), nom_out_img.permute(1,2,0), e2e_out_img.permute(1,2,0)],
         ["Original CIFAR-10",
          f"Nominal (CIFAR)\nPSNR: {nom_psnr_cifar:.2f} | SSIM: {nom_ssim:.2f} | LPIPS: {nom_lpips:.2f}",
          f"End-to-End (CIFAR)\nPSNR: {e2e_psnr_cifar:.2f} | SSIM: {e2e_ssim:.2f} | LPIPS: {e2e_lpips:.2f}"],
         figsize=(13, 5))

    print("Improvements over Nominal Baseline (CIFAR-10):")
    print(f"  PSNR  : {e2e_psnr_cifar - nom_psnr_cifar:+.2f} dB (higher is better)")
    print(f"  SSIM  : {e2e_ssim - nom_ssim:+.2f} (higher is better)")
    print(f"  LPIPS : {e2e_lpips - nom_lpips:+.2f} (lower is better)")

<!-- Merged with Section 8 -->

## 10. Upload & Compare: Nominal vs. End-to-End


In [ ]:
from google.colab import files
from PIL import Image
import io

print("Upload an image to test and compare the trained models:")
uploaded = files.upload()

for fn in uploaded.keys():
    print(f"\nProcessing {fn}...")
    img_bytes = uploaded[fn]
    img = Image.open(io.BytesIO(img_bytes)).convert('RGB')

    # Resize and convert to tensor
    test_transform = T.Compose([T.Resize((IMG, IMG)), T.ToTensor()])
    test_img = test_transform(img).to(device)

    with torch.no_grad():
        # Run through Nominal Pipeline
        nom_out_img, nom_out_psf = pipeline(test_img, nom_optics, nom_isp, EXPOSURE, train=False)
        nom_psnr, nom_ssim, nom_lpips = compute_metrics(nom_out_img, test_img)

        # Run through End-to-End Real Pipeline
        e2e_out_img, e2e_out_psf = pipeline(test_img, e2e_optics_real, e2e_isp_real, EXPOSURE, train=False)
        e2e_psnr, e2e_ssim, e2e_lpips = compute_metrics(e2e_out_img, test_img)

    # Visualize results
    show([test_img.permute(1,2,0), nom_out_img.permute(1,2,0), e2e_out_img.permute(1,2,0)],
         ["Original Upload",
          f"Nominal Baseline\nPSNR: {nom_psnr:.2f} | SSIM: {nom_ssim:.2f} | LPIPS: {nom_lpips:.2f}",
          f"End-to-End (CIFAR)\nPSNR: {e2e_psnr:.2f} | SSIM: {e2e_ssim:.2f} | LPIPS: {e2e_lpips:.2f}"],
         figsize=(13, 5))

    print("Improvements over Nominal Baseline:")
    print(f"  PSNR  : {e2e_psnr - nom_psnr:+.2f} dB (higher is better)")
    print(f"  SSIM  : {e2e_ssim - nom_ssim:+.2f} (higher is better)")
    print(f"  LPIPS : {e2e_lpips - nom_lpips:+.2f} (lower is better)")

Created with the help of Gemini and Claude.